In [1]:
from google.colab import files
uploaded=files.upload()

Saving retail_transactions_100k.csv to retail_transactions_100k.csv


In [3]:
!pip install pyspark

In [5]:
from pyspark.sql import SparkSession
spark=SparkSession.builder \
   .appName("Tranformation_Exercise") \
   .getOrCreate()


In [81]:
df=spark.read.csv("retail_transactions_100k.csv",header=True,inferSchema=True)
df.show(5)

+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+---------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|   product_name|   category|quantity|unit_price|discount_pct|payment_method|   status|rating|
+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+---------+------+
| O000001|2026-02-27|     C00205|Customer_00205|     Pune|  West|      P004|27 Inch Monitor|Electronics|       2|  16961.87|           0|           COD|Delivered|  NULL|
| O000002|2026-04-22|     C01906|Customer_01906|  Lucknow| North|      P010|     Web Camera|Accessories|       1|   4241.16|          20|   Net Banking|Delivered|     1|
| O000003|2026-03-23|     C03463|Customer_03463|Ahmedabad|  West|      P005|   Office Chair|  Furniture|       2|   8590.05|          10|           UP

**Part A — Basic transformations and actions**

1. Filter all transactions where quantity > 5 . Store the result in a new DataFrame without
displaying it.
2. Select only order_id , product_id , quantity , and unit_price . Use show() to
trigger execution.
3. Filter transactions with unit_price > 20000 . Use count() to determine the number of
matching records.
4. Create a new column gross_amount by multiplying quantity * unit_price . Inspect the
execution plan using explain() before displaying the results.
5. Filter transactions with order_status = "Completed" and sort them by unit_price
descending. Trigger execution using take(10) .

In [82]:
from pyspark.sql.functions import col
#1
s1=df.filter(col("quantity")>5)
#2
s2=df.select("order_id","product_id","unit_price")
#
s3=df.filter(col("unit_price")>20000)
print("Total records: ",df.count())
#4
s4=df.withColumn("gross_amount",col("quantity")*col("unit_price"))
s4.explain()
#s4.show()
#5
s5=df.filter(col("status")=="Delivered").orderBy(col("unit_price").desc())
s5.show(3)


Total records:  100000
== Physical Plan ==
*(1) Project [order_id#1828, order_date#1829, customer_id#1830, customer_name#1831, city#1832, region#1833, product_id#1834, product_name#1835, category#1836, quantity#1837, unit_price#1838, discount_pct#1839, payment_method#1840, status#1841, rating#1842, (cast(quantity#1837 as double) * unit_price#1838) AS gross_amount#1924]
+- FileScan csv [order_id#1828,order_date#1829,customer_id#1830,customer_name#1831,city#1832,region#1833,product_id#1834,product_name#1835,category#1836,quantity#1837,unit_price#1838,discount_pct#1839,payment_method#1840,status#1841,rating#1842] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/retail_transactions_100k.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,customer_id:string,customer_name:string,city:string,region...


+--------+----------+-----------+--------------+---------+------+----------+------------+---------

**Part B — Chained transformations**

6. Create a pipeline that filters completed orders, calculates gross amount, selects five relevant
columns, and sorts by gross amount descending. Use only one action at the end.
7. Filter orders with discounts of at least 10%, calculate the discounted sales amount, and display
the top 15 orders.
8. Group transactions by payment_method , calculate total quantity sold, and sort the results in
descending order.
9. Group transactions by sales_channel and order_status , then calculate the transaction
count for each combination.
10. Create a DataFrame containing only order_id , store_id , product_id , and quantity .
Use drop() to remove store_id , then use first() to retrieve one record.

In [23]:
#6
s6 = (
    df.filter(col("status") == "Delivered")
      .withColumn("gross_amount", col("unit_price") * col("quantity"))
      .select("order_id", "customer_id", "unit_price", "quantity", "gross_amount")
      .orderBy(col("gross_amount").desc())
)

s6.show(5)  # the single action

+--------+-----------+----------+--------+------------------+
|order_id|customer_id|unit_price|quantity|      gross_amount|
+--------+-----------+----------+--------+------------------+
| O018458|     C03316|  70183.76|       5|          350918.8|
| O070763|     C01486|  70180.14|       5|          350900.7|
| O011763|     C00514|  70178.65|       5|         350893.25|
| O030561|     C01970|  70177.66|       5|350888.30000000005|
| O097533|     C01899|  70176.03|       5|         350880.15|
+--------+-----------+----------+--------+------------------+
only showing top 5 rows


In [ ]:
#7
s7=(df.filter(col("discount_pct")>=10)
.withColumn("discounted_amt", col("unit_price")*col("discount_pct")/100 ))
s7.show(15)


In [29]:
#8
from pyspark.sql.functions import (min,max,sum,count)
s8=(df.groupBy(col("payment_method"))
.agg(sum("quantity").alias("total_quantity_sold"))
.orderBy(col("total_quantity_sold").desc()))
s8.show()

+--------------+-------------------+
|payment_method|total_quantity_sold|
+--------------+-------------------+
|    Debit Card|              59396|
|   Credit Card|              58843|
|           COD|              58682|
|           UPI|              58448|
|   Net Banking|              58422|
|          NULL|               6056|
+--------------+-------------------+



In [83]:
#9
s9=(df.groupBy("region","status")
.agg(count("*").alias("Transaction Count"))
)
s9.show(3)

+------+----------+-----------------+
|region|    status|Transaction Count|
+------+----------+-----------------+
| South|Processing|             3638|
| North|Processing|             3661|
| North| Cancelled|             2991|
+------+----------+-----------------+
only showing top 3 rows


In [35]:
s10 = (
    df.select("order_id", "customer_id", "product_id", "quantity")
      .drop("store_id")
)

row = s10.first()
print(row)

Row(order_id='O000001', customer_id='C00205', product_id='P004', quantity=2)


\*\***Part C — Joins and Aggregations**\*\*\
\
11\. Join sales_df with products_df using product_id. Select product name, category, quantity, and unit price. Display 20 records.\
\
12\. Join sales_df with stores_df using store_id. Group by city and calculate the total quantity sold.\
\
13\. Join all three DataFrames. Calculate total completed-order revenue for each product category and region.\
\
14\. Filter completed transactions, join the product master, calculate discounted revenue, and display the top five products by total revenue.

In [36]:
uploaded=files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv


In [ ]:
df_prod=spark.read.csv("lazy_execution_products.csv",header=True,inferSchema=True)
df_prod.show(5)

In [44]:
uploaded=files.upload()

Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000 (1).csv


In [ ]:
df_sales=spark.read.csv("lazy_execution_sales_25000.csv",inferSchema=True,header=True)
df_sales.show(5)

In [41]:
uploaded=files.upload()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [ ]:
df_stores=spark.read.csv("lazy_execution_stores.csv",inferSchema=True,header=True)
df_stores.show()

In [ ]:
#11
s11=(df_sales.join(df_prod,on="product_id",how="inner")
.select("product_name", "category", "quantity", "unit_price"))
s11.show(20)

In [49]:
#12
s12=(df_sales.join(df_stores,on="store_id",how="inner")
.groupBy("city")
.agg(sum("quantity").alias("total_quantity_sold"))
)
s12.show()

+---------+-------------------+
|     city|total_quantity_sold|
+---------+-------------------+
|  Chennai|              15727|
|   Mumbai|              15489|
|  Kolkata|              15619|
|     Pune|              15407|
|    Delhi|              15101|
|Bengaluru|              15718|
|Hyderabad|              15784|
|   Jaipur|              16019|
+---------+-------------------+



In [84]:
s13 = (
    df_sales.join(df_prod, on="product_id",how="inner")
            .join(df_stores, on="store_id",how="inner")
            .filter(col("order_status") == "Completed")
            .withColumn("revenue", col("unit_price") * col("quantity"))
            .groupBy("category", "region")
            .agg(sum("revenue").alias("total_revenue"))
)

s13.show(5)

+--------+------+-------------+
|category|region|total_revenue|
+--------+------+-------------+
|Displays| North|     32468090|
|Displays| South|     45761308|
| Storage| South|     36976319|
|Displays|  East|     13616199|
|   Audio|  East|     12043135|
+--------+------+-------------+
only showing top 5 rows


In [56]:
#14
s14=(df_sales.join(df_prod,on="product_id",how="inner")
.filter(col("order_status")=="Completed")
.withColumn("revenue",col("unit_price") * col("quantity") * (1 - col("discount_pct") / 100))
.groupBy("product_name")
.agg(sum("revenue").alias("total_revenue"))
.orderBy(col("total_revenue").desc()) )
s14.show(5)

+---------------+--------------------+
|   product_name|       total_revenue|
+---------------+--------------------+
|      Laptop 16|5.2272546585000014E8|
|      Laptop 14| 3.830051604999999E8|
|  Smartphone 5G| 2.060406317500002E8|
| Android Tablet|1.5040656029999995E8|
|27-inch Monitor|1.1406074015000013E8|
+---------------+--------------------+
only showing top 5 rows


\*\***Part D — Understanding Lazy Execution**\*\*\
\
15\. Create five consecutive transformations using filter(), select(), withColumn(), groupBy(), and orderBy(). Add a Python print() after each transformation. Observe when Spark actually processes the data.\
\
16\. Create a filtered DataFrame and inspect it with explain(). Then call show() and count() separately. Observe the Spark jobs in the Colab output.\
\
17\. Use collect() to retrieve the grouped total quantity for each product category. Explain why collecting a small aggregated result is safer than collecting all 25,000 rows.\
\
18\. Filter completed orders, calculate discounted revenue, and save the resulting DataFrame as Parquet using write.mode("overwrite").parquet(). Identify which operation triggers execution.

In [78]:
#15
from pyspark.sql.functions import lit
s15=df_prod.filter(col("product_name")=="Android Tablet")
print("1. filter done")

s15=df_prod.select("product_name","category")
print("2. select done")

s15=df_prod.withColumn("country",lit("India"))
print("3. withColumn done")

s15=df_prod.groupBy(col("category"))
print("4. groupBy done")

s15=df_prod.orderBy(col("cost_price").desc())
print("5. orderBy done")


1. filter done
2. select done
3. withColumn done
4. groupBy done
5. orderBy done


In [79]:
#16
s16 = df_prod.filter(col("product_name") == "Android Tablet")

s16.explain()   # prints the plan, no job runs
s16.show()      # action: triggers a Spark job
s16.count()     # action: triggers another Spark job

== Physical Plan ==
*(1) Filter (isnotnull(product_name#1038) AND (product_name#1038 = Android Tablet))
+- FileScan csv [product_id#1037,product_name#1038,category#1039,list_price#1040,cost_price#1041] Batched: false, DataFilters: [isnotnull(product_name#1038), (product_name#1038 = Android Tablet)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_products.csv], PartitionFilters: [], PushedFilters: [IsNotNull(product_name), EqualTo(product_name,Android Tablet)], ReadSchema: struct<product_id:string,product_name:string,category:string,list_price:int,cost_price:int>


+----------+--------------+--------+----------+----------+
|product_id|  product_name|category|list_price|cost_price|
+----------+--------------+--------+----------+----------+
|      P008|Android Tablet| Tablets|     22999|     17000|
+----------+--------------+--------+----------+----------+



1

In [74]:
#17
df_sales.join(df_prod, "product_id") \
        .groupBy("category") \
        .agg(sum("quantity").alias("total_quantity")) \
        .collect()

[Row(category='Computers', total_quantity=16258),
 Row(category='Displays', total_quantity=7870),
 Row(category='Mobiles', total_quantity=16572),
 Row(category='Accessories', total_quantity=41839),
 Row(category='Storage', total_quantity=17251),
 Row(category='Audio', total_quantity=17159),
 Row(category='Tablets', total_quantity=7915)]

In [76]:
#18
s18 = (
    df_sales.filter(col("order_status") == "Completed")
            .withColumn("discounted_revenue",
                        col("unit_price") * col("quantity") * (1 - col("discount_pct") / 100))
)

s18.write.mode("overwrite").parquet("/tmp/completed_orders_revenue")